<a href="https://colab.research.google.com/github/hanbiphyun/ESSA_YB/blob/main/ESAA_OB_week5-1_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.kaggle.com/competitions/elo-merchant-category-recommendation/code?competitionId=10445&sortBy=voteCount&excludeNonAccessedDatasources=true

####수상작 리뷰
대회 : Elo Merchant Category Recommendation - 고객의 충성도 예측

특징 : 수천만건의 시계열 거래 데이터를 행렬분해와 피처엔지니어링으로 압축하여 고객 충성도 예측

1. pivot & aggregate : 사용자별로 상점, 구매요일, 구매시간 등을 피봇하여 구체적인 성향을 수치화했다.
2. 얼마나 최근에 샀는지, 얼마나 자주 샀는지를 행렬의 주요 인자로 사용했다.

**특이값 분해 사용**
- 사용자 ID와 상점 ID를 하나의 행렬로 보고 행렬분해수행 -> 잠재요인 벡터를 추출 시 사용자의 취향과 상점 특성이 고차원 벡터로 남는다.
- 이를 통해 LightGBM 모델에 넣었을때 사용자의 구체적 취향을 파악한 정보로 성능을 높임

**이상치 처리**
- 이상치를 제거하는 것이 아니고 이상치일 경우를 예측하는 모델을 생성했다.
- 이진 분류 + 회귀 모형을 사용

**앙상블**
- 스태킹 : LGBM, XGBoost, 여러 NN모델들 쌓음

In [ ]:
import pandas as pd
from sklearn.decomposition import TruncatedSVD

# 1. 유저-상점 카테고리별 거래 횟수 행렬 생성 (Pivot)
user_merchant_matrix = transactions.groupby(['card_id', 'merchant_category_id']).size().unstack(fill_value=0)

# 2. SVD를 이용한 차원 축소 (행렬 분해)
svd = TruncatedSVD(n_components=10, random_state=42)
user_embeddings = svd.fit_transform(user_merchant_matrix)

# 3. 추출된 잠재 요인을 데이터프레임으로 변환 (모델의 입력 피처가 됨)
user_emb_df = pd.DataFrame(user_embeddings, index=user_merchant_matrix.index)
user_emb_df.columns = [f'svd_emb_{i}' for i in range(10)]

In [ ]:
# 유저별 거래 행태를 다각도로 분석하는 집계 함수
agg_func = {
    'purchase_amount': ['sum', 'mean', 'max', 'min', 'std'], # 구매 금액 통계
    'installments': ['sum', 'mean', 'max'],                # 할부 횟수
    'purchase_date': ['max', 'min'],                        # 최근 및 최초 방문일
    'merchant_id': ['nunique'],                             # 얼마나 다양한 상점을 이용하나
    'month_lag': ['mean', 'std']                            # 시간적 분포
}

# card_id(유저) 기준으로 그룹화하여 피처 생성
user_features = transactions.groupby('card_id').agg(agg_func)

# 칼럼명 정리 (ex: purchase_amount_sum)
user_features.columns = ['_'.join(col).strip() for col in user_features.columns.values]

# 방문 주기(Recency) 피처 추가 계산
user_features['diff_days'] = (user_features['purchase_date_max'] - user_features['purchase_date_min']).dt.days